In [ ]:
# gradio is open source library mad eby company owned by huggingface and you cna give some python code to make a react UI.
import gradio as gr 
from dotenv import load_dotenv
import os
from openai import OpenAI
from IPython.display import Markdown, display


In [22]:
load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"


gpt_model = "gpt-4.1-mini"
gemini_model = "gemini-3.1-flash-lite"
ollama_model = "llama3.2:latest"

openai_gpt = OpenAI()
openai_gem = OpenAI(base_url= gemini_url, api_key=google_api_key)
openai_ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama') #didn't end up using this. Used ollama.chat instead



In [ ]:
# when pasing a function to gradio, it is a clalback to that function, to which it can "call back" whenever it wants to
# HTTP tunneling used by share=True
#inbrowser = True: alwasy launches it inthe browser directly, locally ofc
# auth = ("username", "pass") for it to be impoertant to login, helpful with share, you can also pass in a list of userid, password tuples instead of one


In [34]:
system_prompt = "You are a helpful assistant"
from prompt_toolkit.shortcuts import yes_no_dialog

def msg_model(prompt, ai_choice):
    # Select the client and model name based on the UI dropdown choice
    if ai_choice == "GPT":
        client_type = openai_gpt
        model_name = gpt_model
    elif ai_choice == "Gemini":
        client_type = openai_gem
        model_name = gemini_model
    else: # Default to Ollama
        client_type = openai_ollama
        model_name = ollama_model
        
    messages=[{"role":"system", "content":system_prompt}, {"role":"user", "content":prompt}]
    response = client_type.chat.completions.create(model=model_name, messages=messages, stream=True)
    result =""
    for chunk in response:
        result += chunk.choices[0].delta.content or ""
        yield result # using yeild and not return so that graio knows this is a generator and can keep coming back to it 


In [35]:
message_input = gr.Textbox(label="Enter Message:", info="Enter a message for AI to respond to", lines=5)
model_dropdown = gr.Dropdown(choices=["Ollama", "GPT", "Gemini"], value="Ollama", label="Select AI Model")
message_output = gr.Markdown(label="Response:")

gr.Interface(
    fn=msg_model, 
    title="AI Chatbot", 
    inputs=[message_input, model_dropdown], #as many as arguments, always
    outputs=[message_output], 
    examples=[["Hi"], ["What can you do?"]], # Only providing examples for the first input (the message)
    flagging_mode="never"
).launch() #auth = ("me", "me")

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.
